# Chapter 11: 基礎單元測試與四大裝飾器實戰 (Introduction to Unittest)
本章將帶領您掌握 Python 內建的 `unittest` 單元測試框架。透過自動化驗證機制、進階指令參數、條件跳過裝飾器以及生命週期鉤子，我們可以建立全面且穩健的自動化測試環境。

## 1. 什麼是 unittest？
`unittest` 是 Python 內建的單元測試框架（Unit Testing Framework）。
### 🎯 核心用途：
* **自動驗證程式**：不需手動狂按 `print()` 檢查結果。
* **確保函式正確**：給予預期輸入，驗證輸出是否完美符合預期。
* **防範未然**：防止未來修改底層邏輯時，不小心改壞原本正常的功能（防止回歸錯誤 Regression）。
* **例外測試**：驗證程式在遭遇異常時，是否有正確拋出我們指定的 Exception。

## 2. 測試基本結構與三大核心要素
一個標準的測試檔案由以下結構構成：

| 關鍵語法組件 | 實質功能與規範 |
| :--- | :--- |
| **`class TestXXX(unittest.TestCase)`** | **建立測試類別**：必須繼承自 `unittest.TestCase`，讓系統識別這是個測試箱。 |
| **`def test_xxx(self)`** | **測試方法**：方法名稱**必須強制以 `test_` 開頭**，否則測試引擎會直接跳過不執行！ |
| **`self.assertXXX()`** | **驗證條件（斷言）**：用來檢查實際執行結果與預期是否相符。 |
| **`unittest.main()`** | **啟動引擎**：當成主程式執行時，自動掃描並跑完類別內所有 `test_` 開頭的方法。 |

In [ ]:
import unittest

# --- 被測試的目標原始程式 (通常放在獨立的 calc.py) ---
def add(a, b):
    return a + b

def divide(a, b):
    if b == 0:
        raise ZeroDivisionError("除數不能為 0")
    return a / b

# --- 測試程式區塊 (通常放在 test_calc.py) ---
class TestCalc(unittest.TestCase):

    # 測試加法
    def test_add(self):
        self.assertEqual(add(2, 3), 5)
        self.assertNotEqual(add(2, 3), 6)

    # 測試例外狀況是否正確被拋出
    def test_divide_exception(self):
        with self.assertRaises(ZeroDivisionError):
            divide(10, 0)

if __name__ == "__main__":
    # 在 Jupyter Notebook 中執行 unittest 的特殊寫法，argv 是為了避免核心干擾
    unittest.main(argv=['first-arg-is-ignored'], exit=False)

## 3. 常用斷言方法 (Assert Methods) 大補帖
斷言（Assert）是單元測試的核心靈魂，以下為最常用的 10 大斷言工具表：

| 斷言方法 | 檢查的邏輯條件 | 意義說明 |
| :--- | :--- | :--- |
| `assertEqual(a, b)` | `a == b` | 驗證兩者**數值是否相等** |
| `assertNotEqual(a, b)` | `a != b` | 驗證兩者**數值是否不相等** |
| `assertTrue(x)` | `x is True` | 驗證條件結果是否為 **True** |
| `assertFalse(x)` | `x is False` | 驗證條件結果是否為 **False** |
| `assertIs(a, b)` | `a is b` | 驗證兩者是否指向**同一個記憶體物件** |
| `assertIsNot(a, b)` | `a is not b` | 驗證兩者是否為不同的記憶體物件 |
| `assertIsNone(x)` | `x is None` | 驗證目標是否為 **None** |
| `assertIsNotNone(x)` | `x is not None` | 驗證目標是否**不為 None** |
| `assertIn(a, b)` | `a in b` | 驗證 `a` 是否存在於群集 `b` 中 (如串列、字串) |
| `assertNotIn(a, b)` | `a not in b` | 驗證 `a` 是否**不存在**於群集 `b` 中 |

## 4. 終端機命令列執行方式與 `-m` 參數
### 💡 基本執行指令
* `python test_calc.py`：直接將測試檔當作主程式執行。
* `python -m unittest test_calc`：使用 Python 的 unittest 模組來載入並運行測試檔案。`-m` 代表 Module-name (模組名稱)，意思是讓 Python 直接啟動 `unittest` 測試搜集引擎，**即便檔案內沒寫 `unittest.main()` 也能完美執行測試！**

### ⚡ 常用命令列選用參數 (Flags)
* **`-v` (Verbose)**：詳細模式。會印出每個測試方法的名稱與個別的執行結果（FAIL 或 OK）。
* **`-q` (Quiet)**：安靜模式。只印出最後的總結，不顯示細節。
* **`-f` (Failfast)**：速錯模式。只要遇到第一個測試失敗（FAIL）就**立刻中斷**停止，方便除錯。
* **`-c` (Catch)**：控制中斷。當按 `Ctrl+C` 時，會等待目前這一個測試跑完並印出當前結果，而不是直接閃退。
* **`-k` (Pattern)**：過濾篩選。只執行名稱符合特定關鍵字的測試方法。例如：`python -m unittest -k add`

## 5. 測試裝飾器與業界常見應用場景實戰 (Decorators)
以下將針對 4 大測試裝飾器，分別提供獨立且對應真實研發場景的範例代碼。

### 5.1 `@unittest.skip(reason)`
* **業界場景**：專案進入第 2 期，但「第 3 期信用退款功能」底層 API 還在開發中。為了不讓每日自動化流水線（CI/CD）報錯卡住，使用無條件跳過。

In [ ]:
import unittest

class TestFeaturePhase(unittest.TestCase):

    @unittest.skip("【暫時跳過】第3期退款 API 尚未開發完成，預計下週一交付")
    def test_phase3_refund(self):
        # 這裡的程式碼在還沒開發完前，若執行會直接拋出錯誤
        print("這行絕對不會被印出來")
        self.assertEqual(1 + 1, 99)

### 5.2 `@unittest.skipIf(condition, reason)`
* **業界場景**：系統支援 Mac 與 Windows 跨平台，但有些特定行為（例如底層剪貼簿、Windows 註冊表登錄、視窗圖形渲染）只能在 Windows 執行，如果在 macOS 執行測試就會報錯，因此在 Mac 環境下必須跳過（當條件為 True 時跳過）。

In [ ]:
import unittest
import sys

class TestPlatformCompatibility(unittest.TestCase):

    @unittest.skipIf(sys.platform == "darwin", "【環境跳過】macOS 系統不支援 Windows 專屬 Win32 剪貼簿 API")
    def test_windows_clipboard_api(self):
        print(f"正在執行 Windows 核心相容性測試，目前作業系統：{sys.platform}")
        self.assertTrue(True)

### 5.3 `@unittest.skipUnless(condition, reason)`
* **業界場景**：測試對接 LINE Pay 或金流第三方 API 時，必須在環境變數配置了 'LINE_PAY_KEY' 金鑰才能測。除非（Unless）環境變數存在，否則一律跳過，避免其他研發人員因為本機沒有金鑰而無故跑出紅燈失敗（當條件為 False 時跳過）。

In [ ]:
import unittest
import os

# 模擬檢查環境變數中是否存在 API 金鑰 (預設無)
HAS_API_KEY = "LINE_PAY_KEY" in os.environ

class TestThirdPartyIntegration(unittest.TestCase):

    @unittest.skipUnless(HAS_API_KEY, "【依賴跳過】缺乏 LINE_PAY_KEY 環境變數，無法與第三方金流連線測試")
    def test_line_pay_checkout(self):
        print("正在對接線上 LINE Pay 測試沙盒...")
        self.assertTrue(True)

### 5.4 `@unittest.expectedFailure`
* **業界場景**：QA 團隊發現了一個已知的文字解碼錯誤（Bug）。工程師已經寫好測試重現它，但底層核心還沒修復。為了讓測試報告維持綠燈，同時又能持續追蹤進度，掛上這個標記，告訴系統「目前它失敗是預期中的正常現象」（綠燈追蹤）。

In [ ]:
import unittest

class TestBugTracking(unittest.TestCase):

    @unittest.expectedFailure
    def test_text_encoding_bug(self):
        print("-> 正在執行已知轉譯 Bug 的追蹤測試（預期會 Assert 失敗）...")
        # 故意放一個目前必輸的斷言，系統會將其標記為 XFAIL (預期失敗)，不計入紅燈
        self.assertEqual("台灣", "Taiwan_Error", "已知 Bug：字串編碼對接錯誤")

## 6. 測試生命週期鉤子：`setUp` 與 `tearDown`
* **`setUp(self)`**：在**每一個**測試方法執行前自動被呼叫。適合用來產生乾淨的初始資料物件。
* **`tearDown(self)`**：在**每一個**測試方法執行完畢後自動被呼叫。適合用來清空暫存檔或記憶體物件。

In [ ]:
import unittest

class TestLifecycle(unittest.TestCase):

    def setUp(self):
        self.data_center = ["A", "B", "C"]
        print("\n[setUp] 已初始化一組乾淨的測試集")

    def tearDown(self):
        self.data_center.clear()
        print("[tearDown] 已徹底清理環境環境")

    def test_item_exist(self):
        print("  -> 執行測試中：驗證元素存在")
        self.assertIn("B", self.data_center)